In [1]:
#| default_exp large

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'large/pelevin'

In [6]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(full_path, pad_token_id = 50256)
model = GPT2LMHeadModel.from_pretrained(full_path, torch_dtype=torch.bfloat16)
model.config.pad_token_id = model.config.eos_token_id
model.cuda()
model.eval();

In [7]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [8]:
sum(p.numel() for p in model.parameters())

774030080

In [9]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [11]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.37 s, sys: 0 ns, total: 1.37 s
Wall time: 1.34 s


['… Раскопай-ка, Витя, у себя под кителем ямку. Чего это ты на свои миллионы так расплевался? Считай, что они тебе приснились.',
 ' мелкий агент ЦРУ, отравляющий кофе олигархов в лондонской подземке! Кто тебя обучал? Как ты мог так поступить? Это же разрыв сердца, такого не прощают… И зачем ты это сделал? Чтобы произвести впечатление на членов Комитета?',
 ' подхалим. Слыхал про такой грех, наверно? За что получил это от народа?',
 '… А на самом деле ты еще хуже, чем он. Ты просто оттягиваешь неизбежное, как будто тебе есть что скрывать! А я ведь не слепой. Я тебя знаю. Если бы ты хотел умереть, ты бы не стал закрывать глаза.']